In [3]:
import pandas as pd 

df = pd.read_csv("../data/raw/davis_all.csv")

print(df.shape)
print(df.columns.tolist())
print(df.head)
print(df.dtypes)

(30056, 3)
['compound_iso_smiles', 'target_sequence', 'affinity']
<bound method NDFrame.head of                                      compound_iso_smiles  \
0           O=C(NC1CCNCC1)c1[nH]ncc1NC(=O)c1c(Cl)cccc1Cl   
1      CSc1cccc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3...   
2         COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1   
3      COC1C(N(C)C(=O)c2ccccc2)CC2OC1(C)n1c3ccccc3c3c...   
4      CN(C)CC=CC(=O)Nc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2cc...   
...                                                  ...   
30051  Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...   
30052     Cn1cnc2c(F)c(Nc3ccc(Br)cc3Cl)c(C(=O)NOCCO)cc21   
30053                   Cc1ccc2nc(NCCN)c3ncc(C)n3c2c1.Cl   
30054        CC1(C)CNc2cc(NC(=O)c3cccnc3NCc3ccncc3)ccc21   
30055   Cc1cc2c(F)c(Oc3ncnn4cc(OCC(C)O)c(C)c34)ccc2[nH]1   

                                         target_sequence  affinity  
0      MVSYWDTGVLLCALLSCLLLTGSSSGSKLKDPELSLKGTQHIMQAG...  5.000000  
1      MLRGGRRGQLGWHSWAAGPGSLLAWLILASAGAAPCPD

In [30]:
print(df["affinity"].value_counts().head(10))


affinity
5.000000    20931
5.958607      175
5.920819      173
5.886057      141
5.823909      119
5.721246      115
5.853872      113
5.744727      109
5.795880      108
5.769551      100
Name: count, dtype: int64


In [31]:
print(df["affinity"].describe())

count    30056.000000
mean         5.451535
std          0.894717
min          5.000000
25%          5.000000
50%          5.000000
75%          5.522879
max         10.795880
Name: affinity, dtype: float64


In [32]:
print(df["compound_iso_smiles"].nunique())
print(df["target_sequence"].nunique())

68
379


In [33]:
from app.domain.molecule import Molecule
from app.domain.protein import Protein
from app.infrastructure.db.models import MoleculeModel, ProteinModel
from app.infrastructure.db.repositories import MoleculeRepository, ProteinRepository
from app.infrastructure.db.session import async_session_maker


async def insert_molecules(session, df_valid ):
    repository = MoleculeRepository(session)
    smiles_a_id = {}
    
    for smiles in df_valid["compound_iso_smiles"].unique():
        saved = await repository.add(MoleculeModel(smiles=smiles, name=None))
        smiles_a_id[smiles] = saved.id
        
    return smiles_a_id

async def insert_protein(session, df_valid):
   repository = ProteinRepository(session)
   protein_id = {}
   
   for sequence in df_valid["target_sequence"].unique():
       saved = await repository.add(ProteinModel(sequence= sequence, name=None, uniprot_id=None))
       protein_id[sequence] = saved.id
    
   return protein_id
      
  
async def main_pipeline(df: pd.DataFrame):
    
    try:
  
     df["is_valid_molecule"] = [
        Molecule(smiles=iso_smiles, name=None).validate_structure()[0]
        for iso_smiles in df["compound_iso_smiles"]
     ]
     
     df["is_valid_protein"] = [
         Protein(sequence= sequence, name=None, uniprot_id=None).validate_structure()[0]
         for sequence in df["target_sequence"]
     ]
     
    
     df_valid = df[(df["is_valid_molecule"] == True) & (df["is_valid_protein"] == True)]
     
     
    except Exception as val_error:
        print(f"Error durante la fase de validación de datos: {val_error}")
        return
    
    try: 
     async with async_session_maker() as session:
         
        smiles_a_id = await insert_molecules(session, df_valid)
        protein_id = await insert_protein(session, df_valid)
        
        
        print("Pipeline completado con éxito. Datos guardados correctamente.")
    except Exception as db_error:
        print(f"Error en la base de datos. Se realizó un rollback automático: {db_error}")

await main_pipeline(df)

Pipeline completado con éxito. Datos guardados correctamente.
